In [1]:
%pip install llama-index-vector-stores-chroma

Note: you may need to restart the kernel to use updated packages.


In [22]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -U -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [4]:
#importing the dpeendencies
from llama_index.core import Settings  #Set default LLM, embedding model, chunking settings, etc.
from llama_index.core import VectorStoreIndex,SimpleDirectoryReader,StorageContext #Builds the searchable index, Document loading , Connects LlamaIndex with storage/vector DB
from llama_index.vector_stores.chroma import ChromaVectorStore # Stores embeddings
from llama_index.core.node_parser import SimpleNodeParser ##Chunking
from llama_index.embeddings.huggingface import HuggingFaceEmbedding #Text → vectors 
import chromadb #vector db

In [5]:
import nltk

In [6]:
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
docs_dir_path="./docs"
vector_db="./vector_db"
collection_name="documents_collection"

In [8]:
model=HuggingFaceEmbedding()

In [9]:
import os

print(os.listdir(docs_dir_path))

['react.pdf']


In [23]:
from pypdf import PdfReader

reader = PdfReader("./docs/react.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text() or ""

print(text[:2000])

EXPERIMENT-2
 None
None
StudyHub  —  Student  Task  Manager  
(Single-Page
 
App)
 
Scenario  
You  are  building  StudyHub,  a  small  productivity  app  that  helps  students  manage  their  daily  
tasks.
 
This
 
version
 
is
 
a
 
single
 
page
 
—
 
there
 
is
 
no
 
routing
 
and
 
no
 
separate
 
Login/Home/Tasks/Profile
 
pages.
 
Everything
 
lives
 
on
 
one
 
screen,
 
and
 
your
 
job
 
is
 
to
 
focus
 
on
 
state
 
management
 
and
 
performance
 
patterns
:
 
Context
 
API
 
with
 
a
 
custom
 
hook,
 
useReducer
,
 
useMemo
,
 
and
 
React.memo
.
 
Given  Data  
Assume  the  student's  information  is:  
const student = { 
  name: "Priya Nair", 
  email: "priya@gmail.com", 
  year: "3rd Year", 
}; 
Assume  the  student  starts  with  these  tasks:  
const initialTasks = [ 
  { id: 1, title: "Finish DBMS assignment", completed: false }, 
  { id: 2, title: "Revise React hooks", completed: false }, 
  { id: 3, title: "Submit lab report", completed: true }, 
]; None
Compon

In [28]:
from llama_index.core import Document

reader = PdfReader("./docs/react.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text() or ""

documents = [Document(text=text)]

In [ ]:
#loader=SimpleDirectoryReader(input_dir=docs_dir_path)

In [ ]:
#documents=loader.load_data()

In [ ]:
#len(documents)

1

In [ ]:
#print(documents[0])


Doc ID: 9526b8dd-8a09-4932-91a9-bc9ecc38d085
Text: %PDF-1.4 % 1 0 obj <</Title (react) /Producer (Skia/PDF m153
Google Docs Renderer)>> endobj 3 0 obj <</ca 1 /BM /Normal>> endobj 5
0 obj <</Filter /FlateDecode /Length 251>> stream xMj0s@y`]dMEC
hq$I=;pl-$zy]98~Q ʐ|{ ~(DyYJ(<9/{a?0Syy$/W8w%;z?:a  -
FlTkǸ֨M uCJG2ISk?< TC@ LVP;͏lq6 X$6t87xI endstream endobj 10 0
obj <</Filt...


In [29]:
#dividing text into chunks
parser=SimpleNodeParser.from_defaults(chunk_size=500, chunk_overlap=100)
nodes=parser.get_nodes_from_documents(documents)

In [15]:
type(nodes)

list

In [32]:
print(nodes[1].text)

Everything
 
lives
 
on
 
one
 
screen,
 
and
 
your
 
job
 
is
 
to
 
focus
 
on
 
state
 
management
 
and
 
performance
 
patterns
:
 
Context
 
API
 
with
 
a
 
custom
 
hook,
 
useReducer
,
 
useMemo
,
 
and
 
React.memo
.
 
Given  Data  
Assume  the  student's  information  is:  
const student = { 
  name: "Priya Nair", 
  email: "priya@gmail.com", 
  year: "3rd Year", 
}; 
Assume  the  student  starts  with  these  tasks:  
const initialTasks = [ 
  { id: 1, title: "Finish DBMS assignment", completed: false }, 
  { id: 2, title: "Revise React hooks", completed: false }, 
  { id: 3, title: "Submit lab report", completed: true }, 
]; None
Component  Structure  
You  may  follow  this  structure,  or  adapt  it  as  long  as  the  same  responsibilities  are  kept  separate:  
App 
 | 
 |-- StudentProvider 
       | 
       |-- Header   (app title + shows student name via useUser()) 
       | 
       |-- ProfilePanel       (uses useUser() custom hook) 
       | 
       |-- TaskMana

In [33]:
db=chromadb.PersistentClient(path=vector_db) #connect to a ChromaDB database and permanently store its data in ./vector_db

A collection is basically a container inside ChromaDB for related vectors/documents.

In [34]:
chroma_collection=db.get_or_create_collection(name=collection_name) #If this collection already exists, give it to me. Otherwise, create it

ChromaDB = actual vector database
ChromaVectorStore = LlamaIndex's interface to ChromaDB
LlamaIndex
     ↓
ChromaVectorStore
     ↓
ChromaDB

ChromaVectorStore acts like a bridge/adapter.

In [35]:
vector_store=ChromaVectorStore(chroma_collection=chroma_collection) 
#the storage space for embeddings

In [36]:
storage_context=StorageContext.from_defaults(vector_store=vector_store) #For storing my index data, use this vector store
# a wrapper around the vector_Store

In [37]:
#An index is essentially the searchable representation of your document chunks.
index=VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    vector_store=vector_store,
    embed_model=model
)